# AML Part 1 — Colombo District Land Price Prediction

**Objective:** Compare Random Forest and XGBoost regressors for estimating advertised land price per perch in Colombo District, Sri Lanka.

This notebook is intentionally commented for learning and review. Run cells in order from a fresh Google Colab runtime.

**Important data caveat:** The dataset appears to be listing/advertisement data, not confirmed completed transactions. The output is an estimate of advertised price per perch, not a professional valuation.

## 1. Install the required libraries

Colab runtimes are temporary, so install the dependencies at the start of a fresh session.

In [ ]:
%pip -q install xgboost gradio openpyxl joblib

## 2. Imports and reproducibility

We set a random seed so the train/test split and model comparisons can be reproduced.

In [ ]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42
TARGET = "Price per Perch"


## 3. Load the dataset

If the CSV is in the project `data` folder, it is loaded automatically. In Colab, if it is not found, upload `colombo_land_prices.csv` when prompted.

In [ ]:
DATA_PATH = Path("data/colombo_land_prices.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    # Google Colab fallback: upload the CSV included with the submission.
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_names:
            raise ValueError("Please upload colombo_land_prices.csv.")
        df = pd.read_csv(io.BytesIO(uploaded[csv_names[0]]))
    except ImportError:
        raise FileNotFoundError(
            "Could not find data/colombo_land_prices.csv. Put the CSV in the data folder."
        )

print("Original shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())


## 4. Inspect data quality

We inspect types, missing values, exact duplicates, target distribution and the date coverage before cleaning.

In [ ]:
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))
print("Exact duplicate rows:", df.duplicated().sum())
print("Target summary:")
display(df[TARGET].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
if "Posted_Date_new" in df.columns:
    dates = pd.to_datetime(df["Posted_Date_new"], errors="coerce")
    print("Posting date range:", dates.min(), "to", dates.max())
    print("Distinct posting dates:", dates.nunique())


## 5. Clean duplicate rows and define the target

Exact duplicate rows are removed. We do not automatically delete all repeated `Address_ID` values because that field may identify a location, not a unique advertisement.

The target is `Price per Perch`. We explicitly exclude `Mentioned Price(Rs)` and `Price_Scale`, because they are directly related to how the target price was recorded and could leak the answer into the model. `Address_ID` is also excluded because it is an identifier rather than a meaningful measurement.

In [ ]:
before = len(df)
df = df.drop_duplicates().copy()
print(f"Removed {before - len(df)} exact duplicate rows.")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()
df = df[df[TARGET] > 0].copy()
print("Shape after duplicate/target cleaning:", df.shape)


## 6. Exploratory data analysis

Land prices are often right-skewed. The plots below help us understand the distribution and variation across land types. They are descriptive; they do not prove causation.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df[TARGET], bins=50)
plt.xlabel("Price per Perch (LKR)")
plt.ylabel("Number of listings")
plt.title("Distribution of advertised price per perch")
plt.tight_layout()
plt.show()

if "Land_type" in df.columns:
    display(df.groupby("Land_type")[TARGET].agg(["count", "median", "mean"]).sort_values("count", ascending=False).head(15))


## 7. Feature selection and preprocessing

We remove identifiers, the target, and direct price representation fields. The posting date is not used as a raw feature because the data covers only a short date window; the notebook reports that window above.

Numeric columns receive median imputation. Categorical columns receive most-frequent imputation and one-hot encoding. Scaling is included for completeness, although tree models generally do not require feature scaling. All preprocessing is fitted only on the training set through a scikit-learn Pipeline to reduce leakage.

In [ ]:
excluded_columns = [
    TARGET,
    "Mentioned Price(Rs)",  # direct target leakage risk
    "Price_Scale",          # related to how the advertised price is represented
    "Address_ID",           # identifier, not a generalizable property feature
    "Posted_Date_new",      # short posting-date window; not used as raw date
]

feature_columns = [c for c in df.columns if c not in excluded_columns]
X = df[feature_columns].copy()
y = df[TARGET].copy()

# Convert date-derived strings/categories consistently if the source contains them.
for col in X.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns:
    X[col] = X[col].astype(str)

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

print("Feature count:", len(feature_columns))
print("Categorical features:", categorical_features)
print("Numeric features:", len(numeric_features))
print("Excluded columns:", excluded_columns)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2)),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])


## 8. Train/test split

This is an advertisement-level dataset with a relatively short posting period, rather than a long time series for future forecasting. We use a reproducible random holdout as a baseline. If repeated properties/listings are identified, a grouped split would be preferable; do not describe this as a future-market forecast.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 9. Configure and train Random Forest and XGBoost

The key hyperparameters are explicitly set rather than relying on default model settings. These are reasonable starting configurations, not guaranteed optimal values. Further tuning should be justified by validation results.

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=350,
        max_depth=18,
        min_samples_split=6,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBRegressor(
        n_estimators=450,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
print("Both models trained.")


## 10. Evaluate and compare

MAE is the average absolute prediction error in LKR per perch. RMSE penalizes large errors more heavily. R² measures improvement over predicting the test-set mean; it can be negative if a model performs poorly.

Do not select a winner until you have reviewed the actual results. Land prices are skewed, so also interpret the metrics in relation to the median and upper tail.

In [ ]:
def evaluate_regressor(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {
        "Model": name,
        "MAE (LKR/perch)": mean_absolute_error(y_eval, pred),
        "RMSE (LKR/perch)": mean_squared_error(y_eval, pred) ** 0.5,
        "R2": r2_score(y_eval, pred),
    }

results = pd.DataFrame([
    evaluate_regressor("Random Forest", rf_model, X_test, y_test),
    evaluate_regressor("XGBoost", xgb_model, X_test, y_test),
]).sort_values("RMSE (LKR/perch)")
display(results.style.format({
    "MAE (LKR/perch)": "{:,.0f}",
    "RMSE (LKR/perch)": "{:,.0f}",
    "R2": "{:.3f}",
}))


## 11. Interpret the results

Use the actual metric table above to write your interpretation. Address:
- Which model has lower MAE and RMSE?
- Is the R² useful, weak or negative?
- How large is the typical error compared with the median price per perch?
- Could extreme listings, location categories, duplicate/repeated listings or dataset collection bias affect the comparison?
- Random Forest is often robust and less sensitive to some tuning choices; XGBoost can capture complex interactions but may overfit without careful regularization. Relate these general tendencies to your observed results, rather than claiming them as results.

The feature importance section below is only a rough global interpretation. With one-hot encoding, categorical features become multiple encoded columns.

In [ ]:
# Plot observed vs predicted for the model with lower test RMSE.
best_name = results.iloc[0]["Model"]
best_model = rf_model if best_name == "Random Forest" else xgb_model
best_predictions = best_model.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_predictions, alpha=0.45)
limit = max(float(y_test.max()), float(best_predictions.max()))
plt.plot([0, limit], [0, limit], linestyle="--")
plt.xlabel("Actual price per perch (LKR)")
plt.ylabel("Predicted price per perch (LKR)")
plt.title(f"Actual vs predicted — {best_name}")
plt.tight_layout()
plt.show()

print("Model selected by lowest test RMSE:", best_name)


## 12. Save the selected pipeline and create the app

The saved pipeline includes preprocessing and the trained model. The frontend loads this file, so its inputs must use the same feature names and compatible values. The selected model is chosen by test RMSE; record that choice and metrics in your submission.

In [ ]:
# Refit the selected model on all cleaned labelled data for the final demo model.
# Keep the test-set results above as the reported comparison; the final demo model is refitted for inference.
best_model.fit(X, y)
joblib.dump(best_model, "model.pkl")
print("Saved model pipeline to model.pkl")
print("Selected model:", best_name)


## 13. Single-sample inference

Use one held-out test row to demonstrate inference. This example uses the model fitted before the final refit if available; to keep the demonstration honest, we show prediction and actual value from the test set using the previously trained model. The saved model is intended for the app after the final refit.

In [ ]:
# For a clean demonstration of held-out inference, train a demonstration copy on X_train only.
demo_model = rf_model if best_name == "Random Forest" else xgb_model
# Note: best_model has now been refitted on all data, so re-create and fit a fresh demo pipeline.
from sklearn.base import clone
demo_model = clone(rf_model if best_name == "Random Forest" else xgb_model)
demo_model.fit(X_train, y_train)
sample = X_test.iloc[[0]]
actual = float(y_test.iloc[0])
predicted = float(demo_model.predict(sample)[0])
print("Sample input:")
display(sample)
print(f"Actual price per perch: LKR {actual:,.0f}")
print(f"Predicted price per perch: LKR {predicted:,.0f}")
print(f"Absolute error: LKR {abs(actual - predicted):,.0f}")


## 14. Launch the frontend (Gradio)

This creates a temporary public URL for testing. It only works while the Colab runtime is active. The lecturer can run this notebook later in her own Colab session and generate a new link.

The app includes a manual single-property form and a CSV batch-prediction tab.

In [ ]:
# app.py is included in the project. In Colab, upload app.py if it is not in the current directory.
from pathlib import Path
if not Path("app.py").exists():
    try:
        from google.colab import files
        print("Upload app.py from the submitted project.")
        uploaded_app = files.upload()
    except ImportError:
        pass

if Path("app.py").exists():
    # Start the Gradio app in this notebook runtime.
    # The generated share URL is temporary and should not be treated as permanent hosting.
    import subprocess, sys, time, re
    process = subprocess.Popen([sys.executable, "app.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    # Print app output so the share URL appears in the notebook output.
    for _ in range(40):
        line = process.stdout.readline()
        if line:
            print(line.rstrip())
            if "gradio.live" in line or "127.0.0.1" in line:
                break
        else:
            time.sleep(0.5)
else:
    print("app.py not found. Download it from the repository and upload it to Colab, then rerun this cell.")


## 15. Final submission notes

Before submission:
1. Confirm the Kaggle source and license.
2. Review the model comparison and write an interpretation in your own words.
3. Explain that prices are advertised prices and the data is limited to the dataset's collection period.
4. Mention duplicate removal, excluded leakage-prone columns, preprocessing and the chosen hyperparameters.
5. Test the frontend from a fresh runtime.
6. Submit the GitHub repository and notebook. The temporary Gradio link is only a convenience while the runtime is active.